In [3]:
import pandas as pd

In [17]:
def prepare_data(data):
    data = data.copy()

    data.loc[:, "proof_time_s"] = (data["total_proof_time_ms"] / 1000).round(2)

    mask = data["filename"].str.contains("apc0", na=False)
    row = data.loc[mask].drop_duplicates()
    software_proof_time = row["proof_time_s"].squeeze()
    software_proof_time
    
    mask = data["filename"].str.contains("manual", na=False)
    row = data.loc[mask].drop_duplicates()
    manual_proof_time = row["proof_time_s"].squeeze()
    manual_proof_time

    data.loc[:, "faster_than_software"] = (software_proof_time / data["proof_time_s"]).round(2)
    data.loc[:, "slower_than_manual"]   = (data["proof_time_s"] / manual_proof_time).round(2)

    return data[["filename", "num_segments", "proof_time_s", "faster_than_software", "slower_than_manual"]]

In [18]:
data = pd.read_csv("basic_metrics.csv")
data_with_precompiles = pd.read_csv("latest_nightly/reth/basic_metrics.csv")
data_with_precompiles = data_with_precompiles[data_with_precompiles["filename"] == "reth/apc000.json"]
data_with_precompiles["filename"] = "manual"
data = pd.concat([data, data_with_precompiles])
data

,filename,num_segments,app_proof_cells,app_proof_cols,total_proof_time_ms,app_proof_time_ms,app_execute_time_ms,app_trace_gen_time_ms,leaf_proof_time_ms,inner_recursion_proof_time_ms,normal_instruction_ratio,openvm_precompile_ratio,powdr_ratio,powdr_rows
0,metrics_apc0_hints.json,103,222409376923,587384,4139004,2713231,507222,841039,820782,604991,0.964176,0.035824,0.000000,0
1,metrics_apc10_hints.json,28,92685618578,476960,1803486,1106310,435511,1237181,525751,171425,0.653361,0.043143,0.303496,20329984
2,metrics_apc30_hints.json,22,74444765186,545570,1493573,883908,413576,1230015,479678,129987,0.609321,0.051103,0.339576,28705993
3,metrics_apc100_hints.json,17,64280416889,820692,1577348,766001,398140,1315473,710470,100877,0.397418,0.051934,0.550648,39304033
0,manual,6,19622507672,76846,341187,232259,24935,55151,79205,29723,0.594197,0.405803,0.000000,0


In [19]:
prepare_data(data)

,filename,num_segments,proof_time_s,faster_than_software,slower_than_manual
0,metrics_apc0_hints.json,103,4139.00,1.00,12.13
1,metrics_apc10_hints.json,28,1803.49,2.29,5.29
2,metrics_apc30_hints.json,22,1493.57,2.77,4.38
3,metrics_apc100_hints.json,17,1577.35,2.62,4.62
0,manual,6,341.19,12.13,1.00


In [36]:
experiments = ["keccak", "sha256", "ecc", "ecrecover", "u256"]

In [37]:
dfs = {experiment: pd.read_csv(f"{experiment}/basic_metrics.csv") for experiment in experiments}

In [38]:
# Split ECC into two, affine and projective
ecc_data = pd.read_csv("ecc/basic_metrics.csv")

mask = ecc_data["filename"].str.contains("affine|manual", case=False, na=False)
dfs["ecc_affine"] = ecc_data.loc[mask]

mask = ecc_data["filename"].str.contains("projective|manual", case=False, na=False)
dfs["ecc_projective"] = ecc_data.loc[mask]

In [39]:
all_data = {}

for experiment, data in dfs.items():
    data = data.copy()

    data.loc[:, "proof_time_s"] = (data["total_proof_time_ms"] / 1000).round(2)

    mask = data["filename"].str.contains("apc000", na=False)
    row = data.loc[mask].drop_duplicates()
    software_proof_time = row["proof_time_s"].squeeze()
    software_proof_time
    
    mask = data["filename"].str.contains("manual", na=False)
    row = data.loc[mask].drop_duplicates()
    manual_proof_time = row["proof_time_s"].squeeze()
    manual_proof_time

    data.loc[:, "faster_than_software"] = (software_proof_time / data["proof_time_s"]).round(2)
    data.loc[:, "slower_than_manual"]   = (data["proof_time_s"] / manual_proof_time).round(2)

    all_data[experiment] = data[["filename", "num_segments", "proof_time_s",
                                 "faster_than_software", "slower_than_manual"]]

In [40]:
all_data["keccak"]

,filename,num_segments,proof_time_s,faster_than_software,slower_than_manual
0,apc000/metrics.json,49,866.00,1.00,10.05
1,apc003/metrics.json,3,76.88,11.26,0.89
2,apc010/metrics.json,3,78.00,11.10,0.91
3,apc030/metrics.json,3,80.08,10.81,0.93
4,manual/metrics.json,4,86.15,10.05,1.00


In [41]:
all_data["sha256"]

,filename,num_segments,proof_time_s,faster_than_software,slower_than_manual
0,apc000/metrics.json,44,740.50,1.00,15.44
1,apc003/metrics.json,2,115.90,6.39,2.42
2,apc010/metrics.json,2,108.37,6.83,2.26
3,apc030/metrics.json,2,107.59,6.88,2.24
4,manual/metrics.json,2,47.97,15.44,1.00


In [33]:
all_data["ecc_projective"]

,filename,num_segments,proof_time_s,faster_than_software,slower_than_manual
5,manual/metrics.json,1,17.92,62.64,1.00
6,projective-apc000/metrics.json,59,1122.48,1.00,62.64
7,projective-apc003/metrics.json,14,443.56,2.53,24.75
8,projective-apc010/metrics.json,12,453.09,2.48,25.28
9,projective-apc030/metrics.json,4,306.80,3.66,17.12
10,projective-apc100/metrics.json,4,314.53,3.57,17.55


In [34]:
all_data["ecc_affine"]

,filename,num_segments,proof_time_s,faster_than_software,slower_than_manual
0,affine-hint-apc000/metrics.json,24,451.48,1.00,25.19
1,affine-hint-apc003/metrics.json,22,403.70,1.12,22.53
2,affine-hint-apc010/metrics.json,9,266.06,1.70,14.85
3,affine-hint-apc030/metrics.json,2,117.78,3.83,6.57
4,affine-hint-apc100/metrics.json,2,144.05,3.13,8.04
5,manual/metrics.json,1,17.92,25.19,1.00


In [15]:
all_data["ecrecover"]

,filename,num_segments,proof_time_s,faster_than_software,slower_than_manual
0,apc000/metrics.json,34,659.08,1.00,32.31
1,apc003/metrics.json,16,484.46,1.36,23.75
2,apc010/metrics.json,8,341.20,1.93,16.73
3,apc030/metrics.json,5,258.68,2.55,12.68
4,apc100/metrics.json,2,176.67,3.73,8.66
5,manual/metrics.json,1,20.40,32.31,1.00


In [16]:
all_data["u256"]

,filename,num_segments,proof_time_s,faster_than_software,slower_than_manual
0,apc000/metrics.json,34,636.84,1.00,1.72
1,apc003/metrics.json,11,285.02,2.23,0.77
2,apc010/metrics.json,2,83.33,7.64,0.23
3,apc030/metrics.json,2,84.08,7.57,0.23
4,manual/metrics.json,22,369.45,1.72,1.00
